In [ ]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


chat_model = init_chat_model('deepseek-chat')
reason_model = init_chat_model('deepseek-reasoner')


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """choose model based on key word."""
    messages = request.state['messages']
    model = chat_model
    for msg in messages:
        if isinstance(msg, HumanMessage) and 'think' in msg.content:
            model = reason_model
    return handler(request.override(model=model))


agent = create_agent(model=chat_model, middleware=[dynamic_model_selection])

In [2]:
deep_response = agent.invoke(
    {
        'messages': [
            HumanMessage('Please derive the root formula for the cubic equation, think deeply.')
        ]
    }
)

normal_response = agent.invoke(
    {
        'messages': [
            HumanMessage('What is the weather like today?')
        ]
    }
)

In [6]:
deep_msg = deep_response['messages'][-1]
deep_msg.response_metadata

{'token_usage': {'completion_tokens': 3578,
  'prompt_tokens': 17,
  'total_tokens': 3595,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 2696,
   'rejected_prediction_tokens': None},
  'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
  'prompt_cache_hit_tokens': 0,
  'prompt_cache_miss_tokens': 17},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-reasoner',
 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache',
 'id': '05429a2f-bed0-4d18-8440-1c8d608c17a7',
 'finish_reason': 'stop',
 'logprobs': None}

In [7]:
deep_msg.additional_kwargs

{'refusal': None,
 'reasoning_content': 'We are asked: "Please derive the root formula for the cubic equation, think deeply." That means we need to derive Cardano\'s formula for solving a general cubic equation. Let\'s go step by step.\n\nWe consider a general cubic equation:\n\\[\nax^3 + bx^2 + cx + d = 0, \\quad a \\neq 0.\n\\]\nWe can divide by \\(a\\) to get the depressed form. First, let\'s set:\n\\[\nx = y - \\frac{b}{3a}.\n\\]\nThis eliminates the quadratic term. Let\'s do that.\n\nGiven: \\( ax^3 + bx^2 + cx + d = 0 \\). Let \\( x = y - \\frac{b}{3a} \\). Then:\n\\[\nx^3 = \\left(y - \\frac{b}{3a}\\right)^3 = y^3 - 3y^2\\frac{b}{3a} + 3y\\left(\\frac{b}{3a}\\right)^2 - \\left(\\frac{b}{3a}\\right)^3 = y^3 - \\frac{b}{a}y^2 + \\frac{b^2}{3a^2}y - \\frac{b^3}{27a^3}.\n\\]\n\\[\nx^2 = \\left(y - \\frac{b}{3a}\\right)^2 = y^2 - \\frac{2b}{3a}y + \\frac{b^2}{9a^2}.\n\\]\nSubstitute into the equation:\n\\[\na\\left(y^3 - \\frac{b}{a}y^2 + \\frac{b^2}{3a^2}y - \\frac{b^3}{27a^3}\\righ

In [8]:
deep_msg.pretty_print()

================================== Ai Message ==================================

The general cubic equation is:

\[
ax^3 + bx^2 + cx + d = 0, \quad a \neq 0.
\]

The root formula (Cardano's formula) is derived through the following steps:

### 1. Reduce to depressed cubic
Divide by \(a\) to obtain a monic polynomial:

\[
x^3 + \frac{b}{a}x^2 + \frac{c}{a}x + \frac{d}{a} = 0.
\]

Eliminate the quadratic term by substituting \(x = y - \frac{b}{3a}\):

\[
y^3 + py + q = 0,
\]

where

\[
p = \frac{c}{a} - \frac{b^2}{3a^2} = \frac{3ac - b^2}{3a^2}, \quad q = \frac{d}{a} - \frac{bc}{3a^2} + \frac{2b^3}{27a^3} = \frac{27a^2d - 9abc + 2b^3}{27a^3}.
\]

### 2. Vieta's substitution
Set \(y = u + v\), and impose \(uv = -\frac{p}{3}\). Substituting into the depressed cubic:

\[
(u+v)^3 + p(u+v) + q = u^3 + v^3 + 3uv(u+v) + p(u+v) + q = u^3 + v^3 + (3uv + p)(u+v) + q = 0.
\]

With \(uv = -\frac{p}{3}\), the term \(3uv + p = 0\), yielding:

\[
u^3 + v^3 = -q, \quad u^3 v^3 = (uv)^3 = -\frac{p^3}{27

In [9]:
normal_msg = normal_response['messages'][-1]
normal_msg.response_metadata

{'token_usage': {'completion_tokens': 117,
  'prompt_tokens': 11,
  'total_tokens': 128,
  'completion_tokens_details': None,
  'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
  'prompt_cache_hit_tokens': 0,
  'prompt_cache_miss_tokens': 11},
 'model_provider': 'deepseek',
 'model_name': 'deepseek-chat',
 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache',
 'id': '63288566-c01e-44ad-8df4-6b1597b274ec',
 'finish_reason': 'stop',
 'logprobs': None}

In [10]:
normal_msg.additional_kwargs

{'refusal': None}

In [11]:
normal_msg.pretty_print()

================================== Ai Message ==================================

I can’t check real-time weather data directly, but you can easily get today’s weather by:

1. **Searching online** — Try “weather [your city]” in a search engine.  
2. **Using a weather app** — Many smartphones have built-in weather apps.  
3. **Asking a voice assistant** — For example, “Hey Siri” or “OK Google, what’s the weather today?”

If you tell me your city or region, I can help you look up a forecast from a weather website! 🌤️
